# Publication Community → Taxonomy Cluster Mapper

## What this notebook does

Given one or more **communities of publications** (e.g. graph-derived clusters of paper IDs),  
this notebook identifies the **academic taxonomy clusters** that best describe each community.

The pipeline is adapted from `AA-978_nb0006b_journal_market_fit_llm_v2` which maps  
Frontiers journals to taxonomy markets. The same taxonomy, candidate-building, and  
LLM-judgment logic is reused here — the only difference is the unit of analysis:  
instead of a journal, each unit is a **named group of publication IDs you supply**.

### Pipeline overview

```
1. Input            Define communities as {name: [pub_id, ...]}
2. Taxonomy         Load L0/L1/L2 hierarchy, L1 clusters, L2 clusters from BQ
3. Score pull       For each publication, fetch its top-k L2 taxonomy tags
4. Community scope  Count (community, L2) article pairs; filter to in-scope L2s
5. Aggregate        Roll up: L2 → L1 → L1-cluster → L0
6. Candidates       Build a non-overlapping candidate hierarchy per community
7. LLM judgment     GPT-4o assigns each community to core / bleed clusters
8. Flatten          Post-process and deduplicate → df_out
```

### Output

`df_out` — one row per (community, tier, cluster_rank). Columns mirror nb0006b output  
(`community_id`, `community_name`, `tier`, `cluster_rank`, `cluster_key`, `cluster_name`,  
`cluster_level`, `n_l2_covered`, `l2_coverage`, `article_share`, `match_mode`,  
`llm_confidence`, `llm_rationale`, `llm_reasoning`).

No BigQuery writes — save or display however you like at the end.

## 1. Imports and configuration

In [31]:
import json
import logging
import os
import time
from datetime import date
from hashlib import sha256
from textwrap import dedent

import pandas as pd
from google.cloud import bigquery
from openai import OpenAI

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── BigQuery projects ─────────────────────────────────────────────────────────
PROJECT_BILL = "ocean-tech-adv-analytics-c-esf"  # billing project
PROJECT_DATA = "ocean-tech-adv-analytics-c-esf"  # where taxonomy tables live

# ── Source tables ─────────────────────────────────────────────────────────────
TBL_SCORES = f"{PROJECT_DATA}.aa_taxonomy.article_taxonomy_scores_current"
TBL_PUB = "ocean-breeze-tier-1.airak.Publication"  # for pub year lookup
TBL_L2_CLUS = f"{PROJECT_DATA}.aa_taxonomy.l2_cluster_assignments"
TBL_L1_CLUS = f"{PROJECT_DATA}.aa_taxonomy.l1_cluster_assignments"

# ── Scope thresholds (adaptive by community size) ─────────────────────────────
# An L2 topic is "in scope" for a community if it clears EITHER threshold.
# ABS_FLOOR: minimum number of community papers tagged with this L2.
# PCT_FLOOR: minimum share of total community papers (keeps niche L2s for
#            small communities).
SCOPE_ABS_FLOORS = {  # keyed by size_class
    "micro": 2,  # < 50 papers
    "small": 3,  # 50–299
    "medium": 5,  # 300–1999
    "large": 10,  # 2000–9999
    "mega": 15,  # ≥ 10 000
}
SCOPE_PCT_FLOOR = 0.005  # 0.5 % of community papers

# ── L1 depth thresholds ───────────────────────────────────────────────────────
# Controls when an L1 discipline is considered "fully" vs "partially" covered.
L1_DEPTH_FULL = 0.35  # ≥ 35 % of L1's L2 vocabulary in scope → full
L1_DEPTH_PARTIAL = 0.10  # 10–35 % → partial; < 10 % → noise (excluded)

# ── L0 concentration floor ────────────────────────────────────────────────────
# An L1 must carry at least this share of community papers to count toward L0
# coverage. Prevents broad-but-shallow L1 tails from pushing niche communities
# to a spuriously coarse L0 assignment.
L1_ARTICLE_SHARE_FLOOR = 0.010  # 1 %

# ── LLM settings ─────────────────────────────────────────────────────────────
LLM_MODEL = "gpt-4o"
LLM_TEMP = 0.0
MAX_CORE = 4  # maximum core clusters the LLM may select
MAX_BLEED = 5  # maximum bleed clusters

RUN_DATE = date.today().isoformat()

bq = bigquery.Client(project=PROJECT_BILL)
oai = OpenAI(api_key=os.environ.get("GPT4_OPENAI_KEY"))
log.info("Clients ready.  Run date: %s", RUN_DATE)

12:37:10 INFO     Clients ready.  Run date: 2026-06-17


## 2. Input — define your communities

Each community is a named group of publication IDs (integer Microsoft Academic IDs  
as used in the `airak.Publication` table).

```python
communities = {
    "Community A": [123456, 789012, ...],
    "Community B": [...],
}
```

`pub_id` values must match `publication_id` in `aa_taxonomy.article_taxonomy_scores_current`.

> **Large communities (> ~50 k papers):** the BQ query below passes IDs via  
> `UNNEST`, which has a practical limit around 1–2 M values. If your communities  
> are larger, see the comment in Stage 3 for an alternative temp-table approach.

In [32]:
import pandas as pd
from google.cloud import bigquery

# ── BigQuery source tables (update timestamp to match your upload) ────────────
BQ_SRC_PROJECT = "ocean-tech-adv-analytics-c-tfs"
BQ_SRC_DATASET = "scope_drift_raw"
RUN_TIMESTAMP = "20260617_081903"  # <-- update to your actual run timestamp

TBL_CLASSIF = f"{BQ_SRC_DATASET}.classification_raw_20260617_081903"
TBL_PUB_META = f"{BQ_SRC_DATASET}.pub_metadata_raw_20260617_081904"

bq_src = bigquery.Client(project=BQ_SRC_PROJECT)

# Load classification + metadata in a single SQL JOIN
df = bq_src.query(
    f"""
    SELECT 
        c.int_id,
        c.micro,
        c.meso,
        c.macro,
        m.pub_id,
        m.is_frontiers,
        m.journal,
        m.date,
        m.title
    FROM `{TBL_CLASSIF}` c
    JOIN `{TBL_PUB_META}` m
    ON c.int_id = m.int_id
"""
).to_dataframe()
print(f"Loaded {len(df):,} rows from BigQuery")

# Group by macro cluster → communities dict
communities = df.groupby("meso")["pub_id"].apply(list).to_dict()
communities = {f"Cluster {k}": v for k, v in communities.items()}

print(
    f"Communities: {len(communities)}  |  Total pubs: {sum(len(v) for v in communities.values())}"
)

Loaded 55,994 classification rows, 57,141 metadata rows
Communities: 1  |  Total pubs: 55994


In [33]:
# ── Define communities here ──────────────────────────────────────────────────
# Replace with your actual community data.
# Keys are the community labels; values are lists of integer publication IDs.

# communities: dict[str, list[int]] = {
#     "Community 1": [111111, 222222, 333333],  # ← replace with real pub IDs
#     "Community 2": [444444, 555555],
# }

# Internal integer IDs (0-indexed) make downstream lookups easier.
# community_names is the reverse mapping: int ID → label.
community_ids = list(range(len(communities)))
community_names = {i: name for i, name in enumerate(communities.keys())}
community_pubs = {i: pubs for i, (_, pubs) in enumerate(communities.items())}

# Flat list of all pub IDs + a pub_id → community_id mapping for the join below.
all_pub_ids = [
    int(pid) for pubs in community_pubs.values() for pid in pubs
]  # int() for JSON serialization
pub_to_comm = {pid: cid for cid, pubs in community_pubs.items() for pid in pubs}

log.info(
    "Communities: %d  |  total publications: %d", len(communities), len(all_pub_ids)
)
for cid, name in community_names.items():
    log.info("  [%d] %s — %d pubs", cid, name, len(community_pubs[cid]))

12:37:35 INFO     Communities: 1  |  total publications: 55994
12:37:35 INFO       [0] Cluster 0 — 55994 pubs


## 3. Load taxonomy reference data

The taxonomy is a three-level hierarchy:

| Level | Example | Meaning |
|---|---|---|
| L0 domain | Medicine | Broadest grouping (~12 domains) |
| L1 discipline | Immunology | Named academic discipline |
| L2 topic | Autoimmune disease mechanisms | Specific research topic |

**L1 clusters** group related L1 disciplines that frequently co-appear.  
**L2 clusters** group related L2 topics within a single L1.  
These are the same clusters used in the journal market-fit pipeline (nb0006b).  
The taxonomy tables are read-only — nothing is written here.

In [34]:
# ── L2 → L1 → L0 flat mapping ─────────────────────────────────────────────
# l2_cluster_assignments has no l0 columns — join them from l1_cluster_assignments
_df_l2_raw = bq.query(
    f"""
    SELECT DISTINCT l2_taxref AS l2_key,
    l2_name,
    l1_taxref, 
    l1_name
    FROM `{TBL_L2_CLUS}`
"""
).to_dataframe()

_df_l1_l0 = bq.query(
    f"""
    SELECT DISTINCT l1_taxref, l1_name, l0_taxref, l0_name
    FROM `{TBL_L1_CLUS}`
"""
).to_dataframe()

df_l2_map = _df_l2_raw.merge(
    _df_l1_l0[["l1_taxref", "l0_taxref", "l0_name"]].drop_duplicates("l1_taxref"),
    on="l1_taxref",
    how="left",
)
log.info("L2 map: %d rows", len(df_l2_map))
# #region agent log
import json as _json

with open(
    r"c:\Users\sophie.wilson\Documents\scope-drift-model\debug-eeb41f.log", "a"
) as _f:
    _f.write(
        _json.dumps(
            {
                "sessionId": "eeb41f",
                "location": "cell6_df_l2_map",
                "message": "df_l2_map columns",
                "data": {
                    "cols": list(df_l2_map.columns),
                    "sample_l2_key": (
                        str(df_l2_map["l2_key"].iloc[0]) if len(df_l2_map) > 0 else None
                    ),
                    "sample_l2_name": (
                        str(df_l2_map["l2_name"].iloc[0])
                        if "l2_name" in df_l2_map.columns and len(df_l2_map) > 0
                        else "MISSING"
                    ),
                },
                "timestamp": int(__import__("time").time() * 1000),
            }
        )
        + "\n"
    )
# #endregion

# total L2 vocabulary size per L1 (used for depth calculation)
l1_vocab: dict[int, int] = (
    df_l2_map.groupby("l1_taxref")["l2_key"].count().astype(int).to_dict()
)

# ── L1 cluster membership ──────────────────────────────────────────────────
df_l1c_raw = bq.query(
    f"""
    SELECT l0_taxref, l0_name, l1_taxref, l1_name,
           cluster_id, cluster_name, level_index
    FROM `{TBL_L1_CLUS}`
    ORDER BY level_index
"""
).to_dataframe()

# De-duplicate clusters by frozenset of l1_taxrefs
l1c_members: dict[str, frozenset] = {}
l1c_meta: dict[str, dict] = {}
seen_fs: set[frozenset] = set()
_canon = (
    df_l1c_raw.sort_values("level_index")
    .drop_duplicates(["l0_taxref", "cluster_id"])
    .set_index(["l0_taxref", "cluster_id"])["cluster_name"]
)

for (l0_txr, cid), grp in df_l1c_raw.groupby(["l0_taxref", "cluster_id"]):
    members = frozenset(grp["l1_taxref"].tolist())
    if len(members) < 2 or members in seen_fs:
        continue
    seen_fs.add(members)
    key = f"l1c_{sha256(','.join(sorted(str(x) for x in members)).encode()).hexdigest()[:8]}"
    name = _canon.get((l0_txr, cid), f"cluster_{cid}")
    l1c_members[key] = members
    l1c_meta[key] = {
        "key": key,
        "name": name,
        "l0_taxref": int(l0_txr),
        "l0_name": grp["l0_name"].iloc[0],
    }

# Invert: l1_taxref → cluster keys it belongs to
l1_to_clusters: dict[int, list[str]] = {}
for ck, mems in l1c_members.items():
    for t in mems:
        l1_to_clusters.setdefault(int(t), []).append(ck)

# L0 membership: l0_taxref → set of l1_taxrefs
# Built from l1_cluster_assignments which carries l0_taxref + l0_name on every row.
l0_members: dict[int, dict] = {}
for _, r in df_l1c_raw.dropna(subset=["l0_taxref", "l1_taxref"]).iterrows():
    txr = int(r["l0_taxref"])
    if txr not in l0_members:
        l0_members[txr] = {"name": r["l0_name"], "l1s": set()}
    l0_members[txr]["l1s"].add(int(r["l1_taxref"]))

# l0_name → l0_taxref lookup (used in build_candidates to look up cluster l0_taxref)
l0_name_to_taxref: dict[str, int] = {v["name"]: k for k, v in l0_members.items()}

log.info("L1 clusters: %d unique shapes", len(l1c_members))
log.info("L0 domains:  %d", len(l0_members))

# ── L2 cluster membership ──────────────────────────────────────────────────
# Mirrors L1 cluster loading. Clusters group related L2 topics within an L1.
df_l2c_raw = bq.query(
    f"""
    SELECT 
        l1_taxref, 
        l1_name, 
        l2_taxref AS l2_key,
        l2_name, 
        cluster_id, 
        cluster_name
    FROM `{TBL_L2_CLUS}`
    ORDER BY cluster_id
"""
).to_dataframe()

l2c_members: dict[str, frozenset] = {}
l2c_meta: dict[str, dict] = {}
seen_l2fs: set[frozenset] = set()

_l2c_canon = df_l2c_raw.drop_duplicates(["l1_taxref", "cluster_id"]).set_index(
    ["l1_taxref", "cluster_id"]
)["cluster_name"]

for (l1_txr, cid), grp in df_l2c_raw.groupby(["l1_taxref", "cluster_id"]):
    members = frozenset(grp["l2_name"].tolist())
    if len(members) < 2 or members in seen_l2fs:
        continue
    seen_l2fs.add(members)
    key = f"l2c_{sha256(','.join(sorted(members)).encode()).hexdigest()[:8]}"
    name = _l2c_canon.get((l1_txr, cid), f"cluster_{cid}")
    l2c_members[key] = members
    l2c_meta[key] = {
        "key": key,
        "name": name,
        "l1_taxref": int(l1_txr),
        "l1_name": grp["l1_name"].iloc[0],
    }

# Invert: l2_name → cluster keys it belongs to
l2_to_clusters: dict[str, list[str]] = {}
for ck, mems in l2c_members.items():
    for l2 in mems:
        l2_to_clusters.setdefault(l2, []).append(ck)

log.info("L2 clusters: %d unique shapes", len(l2c_members))

c:\Users\sophie.wilson\AppData\Local\miniconda3\envs\scope_drift\lib\site-packages\google\cloud\bigquery\table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
12:37:39 INFO     L2 map: 27524 rows
c:\Users\sophie.wilson\AppData\Local\miniconda3\envs\scope_drift\lib\site-packages\google\cloud\bigquery\table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
12:37:48 INFO     L1 clusters: 338 unique shapes
12:37:48 INFO     L0 domains:  26
c:\Users\sophie.wilson\AppData\Local\miniconda3\envs\scope_drift\lib\site-packages\google\cloud\bigquery\table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
12:38:23 INFO     L2 clusters: 8932 unique shapes


## 4. Pull taxonomy scores for community publications

For every publication in your communities, fetch its **top-k L2 taxonomy tags**  
from `article_taxonomy_scores_current` (`in_top_k = TRUE`, `is_weak_match = FALSE`,  
`level = 2`). These are the same tag filters used in nb0006b.

The result is a flat `(publication_id, l2_key)` table. We join it locally with  
the `pub_to_comm` mapping to produce `(community_id, l2_key, n_articles)`  
— i.e. how many papers in each community are tagged with each L2 topic.

In [35]:
# Pull (pub_id, l2_key) pairs for all community publications from BQ.
# IDs are passed via UNNEST — see the note above if you have > ~1 M pub IDs.
#
# Alternative for very large sets: upload pub_to_comm as a BQ temp table first:
#   df_map = pd.DataFrame(pub_to_comm.items(), columns=["publication_id","community_id"])
#   bq.load_table_from_dataframe(df_map, "your_dataset.tmp_pub_map", ...).result()
# Then JOIN against it in the query below instead of using UNNEST.

df_scores_raw = bq.query(
    f"""
    SELECT
        CAST(t.publication_id AS INT64) AS publication_id,
        t.taxref                       AS l2_key,
        l2.l2_name
    FROM `{TBL_SCORES}` t

    JOIN `{TBL_L2_CLUS}` l2 
    ON t.taxref = l2.l2_taxref

    WHERE t.level         = 2
      AND t.in_top_k      = TRUE
      AND t.is_weak_match = FALSE
      AND CAST(t.publication_id AS INT64) IN UNNEST(@pub_ids)
    """,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[bigquery.ArrayQueryParameter("pub_ids", "INT64", all_pub_ids)]
    ),
).to_dataframe()

log.info(
    "Score rows returned: %d  |  unique pubs matched: %d / %d",
    len(df_scores_raw),
    df_scores_raw["publication_id"].nunique(),
    len(all_pub_ids),
)

# Map publication_id → community_id locally, then aggregate to (community, L2).
df_scores_raw["community_id"] = df_scores_raw["publication_id"].map(pub_to_comm)
df_scores_raw = df_scores_raw.dropna(subset=["community_id"])
df_scores_raw["community_id"] = df_scores_raw["community_id"].astype(int)

df_l2_counts = (
    df_scores_raw.groupby(["community_id", "l2_key"])["publication_id"]
    .nunique()  # count distinct papers per (community, L2) — same as nb0006b
    .reset_index()
    .rename(columns={"publication_id": "n_articles"})
)

# Merge taxonomy hierarchy so each row also carries l1_taxref, l1_name, l0_name.
df_l2_counts = df_l2_counts.merge(
    df_l2_map[["l2_key", "l2_name", "l1_taxref", "l1_name", "l0_taxref", "l0_name"]],
    on="l2_key",
    how="left",
)
unmatched = df_l2_counts["l1_taxref"].isna().sum()
if unmatched:
    log.warning("%d L2 rows could not be matched to taxonomy; dropping", unmatched)
df_l2_counts = df_l2_counts.dropna(subset=["l1_taxref"]).copy()
df_l2_counts["l1_taxref"] = df_l2_counts["l1_taxref"].astype(int)
df_l2_counts["l0_taxref"] = df_l2_counts["l0_taxref"].astype(int)

log.info("(community, L2) pairs after taxonomy join: %d", len(df_l2_counts))
# #region agent log
# import json as _json

# with open(
#     r"c:\\Users\\sophie.wilson\\Documents\\scope-drift-model\\debug-eeb41f.log", "a"
# ) as _f:
#     _f.write(
#         _json.dumps(
#             {
#                 "sessionId": "eeb41f",
#                 "location": "cell8_df_l2_counts",
#                 "message": "df_l2_counts columns",
#                 "data": {
#                     "cols": list(df_l2_counts.columns),
#                     "has_l2_name": "l2_name" in df_l2_counts.columns,
#                 },
#                 "timestamp": int(__import__("time").time() * 1000),
#             }
#         )
#         + "\\n"
#     )
# #endregion

12:40:11 INFO     Score rows returned: 3490154  |  unique pubs matched: 37439 / 55994
12:40:12 INFO     (community, L2) pairs after taxonomy join: 7678


## 5. Build community profiles and in-scope L2 filter

**Profile** — for each community we compute:
- `n_articles`: total publications in the community
- `size_class`: micro / small / medium / large / mega — drives adaptive scope floors
- `age_class` / `growth_class`: publication-year distribution metrics, shown in the LLM brief

Publication years are pulled from the `airak.Publication` table. If your pub IDs  
are not in that table, the profile falls back to size_class only (age/growth = "unknown").

**In-scope filter** — an L2 topic is included for a community if it clears  
**either** the absolute floor (enough papers) **or** the percentage floor  
(even a tiny topic is kept if it's a meaningful share of a small community).  
This mirrors the adaptive filter in nb0006b Stage 2.

In [36]:
from datetime import date as _date

current_year = _date.today().year

# ── Pull publication years for profiling ──────────────────────────────────────
df_pub_years = bq.query(
    f"""
    SELECT
        CAST(p.PublicationId AS INT64) AS publication_id,
        p.PublishedYear                  AS pub_year
    FROM `{TBL_PUB}` p
    WHERE CAST(p.PublicationId AS INT64) IN UNNEST(@pub_ids)
      AND p.PublishedYear IS NOT NULL
    """,
    job_config=bigquery.QueryJobConfig(
        query_parameters=[bigquery.ArrayQueryParameter("pub_ids", "INT64", all_pub_ids)]
    ),
).to_dataframe()

df_pub_years["community_id"] = df_pub_years["publication_id"].map(pub_to_comm)
df_pub_years = df_pub_years.dropna(subset=["community_id"])
df_pub_years["community_id"] = df_pub_years["community_id"].astype(int)

# ── Build profiles ────────────────────────────────────────────────────────────
profiles: dict[int, dict] = {}
for cid in community_ids:
    total = len(community_pubs[cid])

    # Size class
    if total < 50:
        size_class = "micro"
    elif total < 300:
        size_class = "small"
    elif total < 2000:
        size_class = "medium"
    elif total < 10000:
        size_class = "large"
    else:
        size_class = "mega"

    # Age and growth from publication years (if available)
    yr_grp = (
        df_pub_years[df_pub_years["community_id"] == cid]
        .groupby("pub_year")["publication_id"]
        .count()
        .rename("n_articles")
        .reset_index()
        .sort_values("pub_year")
    )

    if not yr_grp.empty:
        yr_grp["cumpct"] = yr_grp["n_articles"].cumsum() / yr_grp["n_articles"].sum()
        start_row = yr_grp[yr_grp["cumpct"] >= 0.10].iloc[0]
        effective_start = int(start_row["pub_year"])
        age = max(1, current_year - effective_start)
        recent = int(yr_grp[yr_grp["pub_year"] >= current_year - 2]["n_articles"].sum())
        early = int(
            yr_grp[yr_grp["pub_year"] <= effective_start + 2]["n_articles"].sum()
        )
        growth_ratio = recent / max(early, 1)

        if age < 3:
            age_class = "new"
        elif age < 7:
            age_class = "growing"
        elif age < 15:
            age_class = "established"
        else:
            age_class = "mature"

        if growth_ratio > 3.0:
            growth_class = "accelerating"
        elif growth_ratio > 1.2:
            growth_class = "growing"
        elif growth_ratio > 0.8:
            growth_class = "stable"
        else:
            growth_class = "declining"
    else:
        effective_start = current_year
        age = growth_ratio = 1
        age_class = growth_class = "unknown"

    profiles[cid] = {
        "community_id": cid,
        "n_articles": total,
        "effective_start": effective_start,
        "age_years": age,
        "growth_ratio": round(growth_ratio, 2),
        "size_class": size_class,
        "age_class": age_class,
        "growth_class": growth_class,
    }
    log.info(
        "[%d] %s — %d pubs  size=%s  age=%s  growth=%s",
        cid,
        community_names[cid],
        total,
        size_class,
        age_class,
        growth_class,
    )


# ── Adaptive in-scope filter ──────────────────────────────────────────────────
# Replace journal_id with community_id in the mark_in_scope helper.
def mark_in_scope(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    abs_floors = df["community_id"].map(
        lambda cid: SCOPE_ABS_FLOORS.get(
            profiles.get(cid, {}).get("size_class", "medium"), 5
        )
    )
    total_arts = df["community_id"].map(
        lambda cid: profiles.get(cid, {}).get("n_articles", 1)
    )
    df["pct"] = df["n_articles"] / total_arts
    df["in_scope"] = (df["n_articles"] >= abs_floors) | (df["pct"] >= SCOPE_PCT_FLOOR)
    return df[df["in_scope"]].drop(columns=["pct", "in_scope"])


df_scope = mark_in_scope(df_l2_counts)
# Rename community_id column to journal_id so downstream functions (copied from
# nb0006b and keyed on journal_id) work without modification.
df_scope = df_scope.rename(columns={"community_id": "journal_id"})
log.info(
    "In-scope: %d (community, L2) pairs  (from %d raw)",
    len(df_scope),
    len(df_l2_counts),
)

c:\Users\sophie.wilson\AppData\Local\miniconda3\envs\scope_drift\lib\site-packages\google\cloud\bigquery\table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
12:40:19 INFO     [0] Cluster 0 — 55994 pubs  size=mega  age=mature  growth=growing
12:40:19 INFO     In-scope: 1870 (community, L2) pairs  (from 7678 raw)


## 6. Aggregate L2 → L1 → L1-cluster → L0

Exactly the same aggregation as nb0006b Stages 3 & 4. The `journal_id` column  
name is kept internally so the functions copy over without change — it refers  
to `community_id` throughout.

**L1 aggregation** computes, per (community, L1 discipline):
- `n_scope_l2`: how many of the L1's L2 topics are in scope for this community
- `depth`: `n_scope_l2 / l1_vocab_size` — how thoroughly the community covers the discipline
- `depth_class`: `full` (≥ 35 %) / `partial` (10–35 %) / `noise` (< 10 %, excluded)

**L1-cluster aggregation** checks which discipline clusters the community covers  
across multiple L1s — a cluster is surfaced only if ≥ 50 % of its L1s are present.

**L0 aggregation** surfaces broad domain-level membership, with a concentration  
check to prevent journals with a thin long tail from being assigned to L0.

In [37]:
# ── L1 aggregation ────────────────────────────────────────────────────────
# For each (journal, L1): how many in-scope L2s, and what fraction of the L1 vocabulary?
df_l1_agg = (
    df_scope.groupby(["journal_id", "l1_taxref", "l1_name", "l0_taxref", "l0_name"])
    .agg(n_scope_l2=("l2_key", "count"), n_articles_in_l1=("n_articles", "sum"))
    .reset_index()
)
df_l1_agg["l1_vocab_size"] = df_l1_agg["l1_taxref"].map(l1_vocab).fillna(1).astype(int)
df_l1_agg["depth"] = df_l1_agg["n_scope_l2"] / df_l1_agg["l1_vocab_size"]

# Classify depth
df_l1_agg["depth_class"] = pd.cut(
    df_l1_agg["depth"],
    bins=[-1, L1_DEPTH_PARTIAL, L1_DEPTH_FULL, 2.0],
    labels=["noise", "partial", "full"],
)

# Keep only meaningful L1 presence (drop noise tier)
df_l1_sig = df_l1_agg[df_l1_agg["depth_class"] != "noise"].copy()
log.info("Significant (journal, L1) pairs: %d", len(df_l1_sig))

# ── L1-cluster aggregation ─────────────────────────────────────────────────
# For each journal, check which L1 clusters have meaningful presence.
# A cluster is "in scope" if ≥ half its constituent L1s are significant.
cluster_rows = []
for jid, jgrp in df_l1_sig.groupby("journal_id"):
    sig_l1s = set(jgrp["l1_taxref"].astype(int))
    full_l1s = set(jgrp[jgrp["depth_class"] == "full"]["l1_taxref"].astype(int))
    for ck, mems in l1c_members.items():
        active = mems & sig_l1s
        full = mems & full_l1s
        if not active:
            continue
        frac_active = len(active) / len(mems)
        frac_full = len(full) / len(mems)
        # only surface cluster if majority of its L1s are present
        if frac_active < 0.5:
            continue
        # articles in cluster = sum of articles across member L1s with scope data
        art_in_cluster = int(
            jgrp[jgrp["l1_taxref"].isin(active)]["n_articles_in_l1"].sum()
        )
        cluster_rows.append(
            {
                "journal_id": int(jid),
                "cluster_key": ck,
                "cluster_name": l1c_meta[ck]["name"],
                "l0_taxref": l1c_meta[ck]["l0_taxref"],
                "l0_name": l1c_meta[ck]["l0_name"],
                "n_l1s": len(mems),
                "n_active_l1s": len(active),
                "n_full_l1s": len(full),
                "frac_active": round(frac_active, 3),
                "frac_full": round(frac_full, 3),
                "n_articles": art_in_cluster,
            }
        )

df_clusters_agg = pd.DataFrame(cluster_rows)
log.info("(journal, L1-cluster) pairs in scope: %d", len(df_clusters_agg))

# ── L0 aggregation ─────────────────────────────────────────────────────────
l0_rows = []
for jid, jgrp in df_l1_sig.groupby("journal_id"):
    sig_l1s = set(jgrp["l1_taxref"].astype(int))
    full_l1s = set(jgrp[jgrp["depth_class"] == "full"]["l1_taxref"].astype(int))
    # Concentration check: only count an L1 toward effective L0 coverage if it
    # carries at least L1_ARTICLE_SHARE_FLOOR of the journal's total articles.
    # This stops journals with a focused core + long sparse tail from being
    # pushed to L0 when a tighter L1 cluster would be a better fit.
    j_total_arts = jgrp["n_articles_in_l1"].sum()
    concentrated_l1s = set(
        jgrp[jgrp["n_articles_in_l1"] / j_total_arts >= L1_ARTICLE_SHARE_FLOOR][
            "l1_taxref"
        ].astype(int)
    )

    for l0_txr, l0 in l0_members.items():
        active = l0["l1s"] & concentrated_l1s  # ← was sig_l1s
        active_any = l0["l1s"] & sig_l1s  # unweighted — for n_active_l1s reporting
        full = l0["l1s"] & full_l1s
        if not active:
            continue
        frac_active = len(active) / len(l0["l1s"])
        art_in_l0 = int(
            jgrp[jgrp["l1_taxref"].isin(active_any)]["n_articles_in_l1"].sum()
        )
        l0_rows.append(
            {
                "journal_id": int(jid),
                "l0_taxref": l0_txr,
                "l0_name": l0["name"],
                "n_l1s": len(l0["l1s"]),
                "n_active_l1s": len(active_any),
                "n_full_l1s": len(full),
                "frac_active": round(frac_active, 3),  # now concentration-weighted
                "n_articles": art_in_l0,
            }
        )

df_l0_agg = pd.DataFrame(l0_rows)
log.info("(journal, L0) pairs in scope: %d", len(df_l0_agg))

12:40:19 INFO     Significant (journal, L1) pairs: 187
12:40:19 INFO     (journal, L1-cluster) pairs in scope: 70
12:40:19 INFO     (journal, L0) pairs in scope: 8


## 7. Build non-overlapping candidate hierarchy

`build_candidates` constructs the structured **brief** that is sent to the LLM  
for each community. It follows a strict hierarchy to prevent overlap:

- **Step A** — L1 clusters first: surface clusters where ≥ 2 member disciplines  
  are fully covered. Mark all their L1 members as "spoken for".
- **Step B** — L1 singles: only for disciplines NOT already inside a surfaced cluster.
- **Step C** — L0 domains: only when ≥ 2 L1 clusters exist within the same L0.
- **Step D** — L2 cluster fallback: fires for micro/small communities with no  
  coherent L1 signal — the LLM assesses at topic-cluster level instead.

The hierarchy guarantees the LLM never sees two candidates describing the same  
L2 territory, which avoids double-counting in its assignments.

In [38]:
def build_candidates(journal_id: int) -> dict:
    """
    Return a structured, overlap-free candidate brief for one journal.

    Hierarchy (coarsest → finest):
      L0 domains  →  L1 clusters (within each L0)  →  L1 singles  →  in-scope L2s

    Overlap is prevented by construction:
    - L1 singles are only surfaced for L1s NOT already fully represented by an L1 cluster.
    - L1 clusters are only surfaced where they add coverage beyond individual L1s.
    - L0 domains are only surfaced where they encompass ≥2 active L1 clusters.

    The resulting brief is the input to the LLM — it never sees two candidates
    that describe the same L2 territory.
    """
    profile = profiles.get(journal_id, {})
    l1_rows = df_l1_sig[df_l1_sig["journal_id"] == journal_id]
    clus_rows = df_clusters_agg[df_clusters_agg["journal_id"] == journal_id]
    l0_rows_j = df_l0_agg[df_l0_agg["journal_id"] == journal_id]

    sig_l1_set = set(l1_rows["l1_taxref"].astype(int))
    full_l1_set = set(
        l1_rows[l1_rows["depth_class"] == "full"]["l1_taxref"].astype(int)
    )

    # Track which L1 taxrefs are "spoken for" by a surfaced cluster
    covered_by_cluster: set[int] = set()

    # Step A: select L1 clusters to surface.
    # A cluster is surfaced if it has ≥2 full L1s — meaning it genuinely covers
    # a coherent sub-domain rather than just catching a journal's one L1.
    surfaced_clusters = []
    for _, cr in clus_rows[clus_rows["n_full_l1s"] >= 2].iterrows():
        mems = l1c_members[cr["cluster_key"]]
        full_in_cluster = mems & full_l1_set
        surfaced_clusters.append(
            {
                "level": "l1_cluster",
                "key": cr["cluster_key"],
                "name": cr["cluster_name"],
                "l0_name": cr["l0_name"],
                "n_l1s": cr["n_l1s"],
                "n_full_l1s": cr["n_full_l1s"],
                "frac_full": cr["frac_full"],
                "n_articles": cr["n_articles"],
                "member_l1s": [
                    l1_rows[l1_rows["l1_taxref"] == t]["l1_name"].iloc[0]
                    for t in full_in_cluster
                    if len(l1_rows[l1_rows["l1_taxref"] == t]) > 0
                ],
            }
        )
        covered_by_cluster |= mems  # mark ALL cluster L1s as covered

    # Step B: surface L1 singles for L1s NOT covered by any surfaced cluster.
    # Also surface them if the journal covers them deeply but the cluster wasn't
    # surfaced (e.g. the journal is a pure single-L1 niche journal).
    surfaced_l1s = []
    for _, lr in l1_rows[~l1_rows["l1_taxref"].isin(covered_by_cluster)].iterrows():
        # Pull top-5 in-scope L2s for this L1 (by article count)
        l2s_for_l1 = (
            df_scope[
                (df_scope["journal_id"] == journal_id)
                & (df_scope["l1_taxref"] == lr["l1_taxref"])
            ]
            .sort_values("n_articles", ascending=False)
            .head(5)[["l2_key", "l2_name", "n_articles"]]
            .to_dict("records")
        )
        surfaced_l1s.append(
            {
                "level": "l1_single",
                "key": f"l1_{int(lr['l1_taxref'])}",
                "name": lr["l1_name"],
                "l0_name": lr["l0_name"],
                "depth": round(float(lr["depth"]), 3),
                "depth_class": str(lr["depth_class"]),
                "n_scope_l2": int(lr["n_scope_l2"]),
                "l1_vocab_size": int(lr["l1_vocab_size"]),
                "n_articles": int(lr["n_articles_in_l1"]),
                "top_l2s": l2s_for_l1,
            }
        )

    # Step C: surface L0 domains where ≥2 clusters are present.
    # An L0 candidate subsumes its constituent clusters — the LLM can pick the
    # L0 instead of listing all clusters individually.
    surfaced_l0s = []
    for _, lr in l0_rows_j[l0_rows_j["frac_active"] >= 0.5].iterrows():
        # How many surfaced clusters belong to this L0?
        clusters_in_l0 = [c for c in surfaced_clusters if c["l0_name"] == lr["l0_name"]]
        if len(clusters_in_l0) < 2:
            continue
        surfaced_l0s.append(
            {
                "level": "l0_domain",
                "key": f"l0_{int(lr['l0_taxref'])}",
                "name": lr["l0_name"],
                "n_l1s_total": lr["n_l1s"],
                "n_active_l1s": lr["n_active_l1s"],
                "n_full_l1s": lr["n_full_l1s"],
                "frac_active": lr["frac_active"],
                "n_articles": lr["n_articles"],
                "clusters": [c["name"] for c in clusters_in_l0],
            }
        )

    # Step D: L2 cluster fallback.
    # Fires when the journal has no coherent L1 presence — either because there
    # is no L1 signal at all, or because it is a small/niche journal spread
    # thinly across many partial L1s (the cross-disciplinary niche pattern).
    # In the latter case we also clear the L1 singles so the LLM isn't confused
    # by two competing levels of granularity.
    is_niche = (
        not surfaced_l0s
        and not surfaced_clusters
        and (
            not surfaced_l1s
            or (
                profile.get("size_class") in ("micro", "small")
                and all(s["depth_class"] == "partial" for s in surfaced_l1s)
            )
        )
    )
    l2_clusters_out = []
    l2_fallback = False
    if is_niche:
        l2_rows = df_scope[df_scope["journal_id"] == journal_id]
        if not l2_rows.empty:
            l2_fallback = True
            surfaced_l1s = []  # clear L1 singles — L2 clusters take precedence
            jl2_keys = set(l2_rows["l2_key"].tolist())

            for ck, members in l2c_members.items():
                overlap = members & jl2_keys
                if not overlap:
                    continue
                art_count = int(
                    l2_rows[l2_rows["l2_key"].isin(overlap)]["n_articles"].sum()
                )
                meta = l2c_meta[ck]
                l2_clusters_out.append(
                    {
                        "level": "l2_cluster",
                        "key": ck,
                        "name": meta["name"],
                        "l1_name": meta["l1_name"],
                        "n_l2s_total": len(members),
                        "n_l2s_covered": len(overlap),
                        "frac_covered": round(len(overlap) / len(members), 3),
                        "n_articles": art_count,
                        "member_l2s": sorted(overlap)[:6],
                    }
                )

            l2_clusters_out.sort(key=lambda x: -x["n_articles"])

    return {
        "journal_id": journal_id,
        "journal_name": community_names.get(journal_id, str(journal_id)),
        "profile": profile,
        "l0_domains": surfaced_l0s,
        "l1_clusters": surfaced_clusters,
        "l1_singles": surfaced_l1s,
        "l2_clusters": l2_clusters_out,
        "n_scope_l2_total": int(
            df_scope[df_scope["journal_id"] == journal_id].shape[0]
        ),
        "_l2_fallback": l2_fallback,
    }


# Build briefs for all journals
# Build the candidate brief for every community.
# community_ids maps to the same integer keys used in profiles and df_scope.
journal_ids = (
    community_ids  # re-use the variable name so build_candidates works unchanged
)
briefs: dict[int, dict] = {cid: build_candidates(cid) for cid in journal_ids}
log.info("Candidate briefs built for %d journals", len(briefs))

12:40:19 INFO     Candidate briefs built for 1 journals


## 8. LLM judgment — assign clusters to communities

`format_brief` converts the candidate hierarchy into a structured prompt.  
`call_llm` sends it to GPT-4o and handles hallucinated-key correction  
(up to 2 clarification rounds) before falling back to dropping bad keys.

The system prompt is adapted from nb0006b — "journal" is replaced with  
"publication community" so the LLM understands the unit of analysis.  
All selection rules (core/bleed distinction, no-overlap constraint,  
granularity guidance) are identical.

In [39]:
def format_brief(brief: dict, journal_name: str) -> str:
    """Convert a candidate brief into the user-turn text sent to the LLM."""
    p = brief["profile"]
    lines = [
        f"## Publication community: {journal_name}",
        f"Profile: {p.get('size_class','?')} ({p.get('n_articles','?')} papers total), "
        f"{p.get('age_class','?')} community ({p.get('age_years','?')} yrs), "
        f"growth: {p.get('growth_class','?')} (×{p.get('growth_ratio','?')} recent vs early)",
        f"Total in-scope L2 topics: {brief['n_scope_l2_total']}",
    ]
    if brief.get("_l2_fallback"):
        lines.append(
            "⚠ This is a new or very small journal with no significant L1-level signal. "
            "Its market position should be assessed at L2 topic granularity. "
            "Select from the L2 topic candidates below — do NOT force an L1 selection."
        )
    lines.append("")

    if brief["l0_domains"]:
        lines.append(
            "### L0 Domain candidates (broadest — pick if journal spans a whole domain)"
        )
        for d in sorted(brief["l0_domains"], key=lambda x: -x["n_articles"]):
            lines.append(
                f"  key={d['key']}  name='{d['name']}'  "
                f"covers {d['n_active_l1s']}/{d['n_l1s_total']} sub-disciplines  "
                f"({d['frac_active']:.0%} active, {d['n_full_l1s']} fully covered)  "
                f"articles={d['n_articles']:,}  "
                f"sub-clusters: {', '.join(d['clusters'][:4])}"
                + (" …" if len(d["clusters"]) > 4 else "")
            )
        lines.append("")

    if brief["l1_clusters"]:
        lines.append(
            "### L1 Cluster candidates (pick if journal spans a coherent group of disciplines)"
        )
        for c in sorted(brief["l1_clusters"], key=lambda x: -x["n_articles"]):
            lines.append(
                f"  key={c['key']}  name='{c['name']}'  "
                f"L0='{c['l0_name']}'  "
                f"{c['n_full_l1s']}/{c['n_l1s']} disciplines fully covered  "
                f"articles={c['n_articles']:,}  "
                f"disciplines: {', '.join(c['member_l1s'][:4])}"
                + (" …" if len(c["member_l1s"]) > 4 else "")
            )
        lines.append("")

    if brief["l1_singles"]:
        lines.append(
            "### L1 Single-discipline candidates (pick for focused/niche journals)"
        )
        for s in sorted(brief["l1_singles"], key=lambda x: -x["n_articles"]):
            # #region agent log
            import json as _json

            if s.get("top_l2s"):
                with open(
                    r"c:\\Users\\sophie.wilson\\Documents\\scope-drift-model\\debug-eeb41f.log",
                    "a",
                ) as _f:
                    _f.write(
                        _json.dumps(
                            {
                                "sessionId": "eeb41f",
                                "location": "format_brief_top_l2s",
                                "message": "top_l2s structure",
                                "data": {
                                    "first_item": (
                                        s["top_l2s"][0] if s["top_l2s"] else None
                                    ),
                                    "has_l2_name": (
                                        "l2_name" in s["top_l2s"][0]
                                        if s["top_l2s"]
                                        else False
                                    ),
                                },
                                "timestamp": int(__import__("time").time() * 1000),
                            }
                        )
                        + "\\n"
                    )
            # #endregion
            top = ", ".join(t["l2_name"].split(" - ", 1)[-1] for t in s["top_l2s"][:3])
            lines.append(
                f"  key={s['key']}  name='{s['name']}'  "
                f"L0='{s['l0_name']}'  "
                f"depth={s['depth']:.0%} ({s['n_scope_l2']}/{s['l1_vocab_size']} topics)  "
                f"articles={s['n_articles']:,}  "
                f"top topics: {top}"
            )
        lines.append("")

    if brief.get("l2_clusters"):
        lines.append(
            "### L2 Cluster candidates (niche journal — assess at topic-cluster level)"
        )
        for c in brief["l2_clusters"]:
            lines.append(
                f"  key={c['key']}  name='{c['name']}'  "
                f"parent L1='{c['l1_name']}'  "
                f"covers {c['n_l2s_covered']}/{c['n_l2s_total']} topics "
                f"({c['frac_covered']:.0%})  articles={c['n_articles']:,}  "
                f"topics: {', '.join(t.split(' - ', 1)[-1] for t in c['member_l2s'][:4])}"
                + (" …" if len(c["member_l2s"]) > 4 else "")
            )
        lines.append("")

    return "\n".join(lines)


def parse_llm_response(text: str) -> dict:
    """Extract the JSON block from the LLM's response."""
    import re

    match = re.search(r"```json\s*([\s\S]+?)\s*```", text)
    if match:
        return json.loads(match.group(1))
    # Fallback: try last {...} block
    match = re.search(r"(\{[\s\S]+\})", text)
    if match:
        return json.loads(match.group(1))
    raise ValueError("No JSON found in LLM response")


SYSTEM_PROMPT = dedent(
    """
    You are an expert in academic publishing and research taxonomy.
    Your task: determine the **academic topic position** of a publication community based on
    its publication data — i.e., which topic cluster(s) best describe what the community
    covers as its primary focus.

    You will receive a structured brief containing:
    - A community profile (size, age, growth trajectory based on publication years)
    - Candidate taxonomy clusters at multiple levels of granularity (L0 domain,
      L1 cluster, L1 single discipline) derived from its publication record
    - Each candidate shows how many of the community's papers fall within it

    ## Key format
    Every candidate has a key field in one of these exact formats:
      l0_<integer>     — a broad domain  (e.g. l0_86903686)
      l1c_<8hexchars>  — a discipline cluster  (e.g. l1c_3a7f912b)
      l1_<integer>     — a single discipline  (e.g. l1_2762788928)
      l2c_<8hexchars>  — a topic cluster (e.g. l2c_7d3a1b9c)
                         Only present for small/niche journals assessed at topic-cluster level.
    You MUST copy keys character-for-character from the brief.
    Do NOT invent, abbreviate, or paraphrase keys.

    ## Rules
    1. Select 1–{max_core} **core** clusters that together define the journal's
       primary market. Core = what the journal is fundamentally *about*.
       **An empty core is almost always wrong.** Even for cross-disciplinary journals
       that span multiple fields, the clusters defining the journal's primary subject
       area belong in core — not bleed.
       When no single cluster dominates (i.e. `match_mode` = "combination"), the
       selected combination of clusters IS the core market. Bleed is reserved for
       topics genuinely adjacent to that combination — not for the combination itself.
    2. Optionally select 1–{max_bleed} **bleed** clusters for secondary/adjacent
       markets the journal meaningfully covers but that are not its defining focus.
       **A second (or further) core cluster requires ≥ 25 % article share.**
       If a cluster accounts for less than 25 % of the journal's articles, it belongs
       in bleed — not core — regardless of how broad or prominent it appears.
       Note: ≥ 25 % is a *minimum threshold*, not a promotion rule. A cluster above
       25 % that is adjacent or secondary (not a defining focus) should still be bleed.
       Use judgment: core = what the journal is fundamentally *about*.
       **Never leave core empty** when valid candidates exist — if the journal's
       primary field is not a dominant single cluster, put the combination in core.
    3. **No overlap**: never select both a parent and one of its children
       (e.g. do not select both "Medicine" L0 and "Ophthalmology" L1 — the L0
       already contains the L1). If a parent is selected, it represents all children.
    4. **Granularity guidance**:
       - If a journal covers ≥ 50 % of an L0 domain's sub-disciplines → prefer L0
       - If a journal covers a coherent cluster of 2+ L1 disciplines deeply → prefer L1-cluster
       - If a journal is clearly a single-discipline journal → prefer L1-single
       - If the brief shows **L2 cluster candidates** (⚠ warning present) → select from
         those `l2c_` keys only. Do NOT select L1 or L0 keys for these journals. Their
         market position is at topic-cluster level, not discipline level.
    5. For **bleed**: choose clusters that represent the journal's secondary markets
       or adjacent fields — areas it publishes in but which are not its main identity.
       These must not overlap with each other or with core selections.
    6. Return your reasoning FIRST, then a JSON block. The JSON must be the last
       thing in your response, in this exact format:

    ```json
    {{
      "core": [
        {{"key": "...", "name": "...", "level": "...", "rationale": "..."}}
      ],
      "bleed": [
        {{"key": "...", "name": "...", "level": "...", "rationale": "..."}}
      ],
      "match_mode": "primary | combination",
      "confidence": "high | medium | low",
      "overall_reasoning": "..."
    }}
    ```

    `match_mode` is "primary" if a single core cluster accounts for ≥ 60 % of
    the journal's publications; otherwise "combination".
    Only use keys that appear in the brief — do not invent new ones.
"""
).strip()


CLARIFY_TEMPLATE = dedent(
    """
    Your previous response contained {n} selection(s) with keys that do not exist
    in the brief I provided:

    {bad_lines}

    This means you either paraphrased or invented those keys.
    Keys must be copied exactly as they appear in the brief.

    The complete list of valid keys for this journal is:
    {valid_key_lines}

    Please reselect replacements for the invalid entries only, choosing the
    closest match from the valid keys above.

    Return ONLY the corrected JSON block (no additional prose) in this format:
    ```json
    {{
      "core": [...],
      "bleed": [...],
      "match_mode": "...",
      "confidence": "...",
      "overall_reasoning": "..."
    }}
    ```
"""
).strip()


def _valid_keys_from_brief(brief: dict) -> set[str]:
    keys = set()
    for d in brief.get("l0_domains", []):
        keys.add(d["key"])
    for c in brief.get("l1_clusters", []):
        keys.add(c["key"])
    for s in brief.get("l1_singles", []):
        keys.add(s["key"])
    for c in brief.get("l2_clusters", []):
        keys.add(c["key"])
    return keys


def _hallucinated(selections: list[dict], valid_keys: set[str]) -> list[dict]:
    return [s for s in selections if s.get("key") not in valid_keys]


def _build_clarify_msg(data: dict, valid_keys: set[str], brief: dict) -> str:
    bad = _hallucinated(data.get("core", []), valid_keys) + _hallucinated(
        data.get("bleed", []), valid_keys
    )
    bad_lines = "\n".join(
        f"  - key='{s.get('key')}' name='{s.get('name')}' (tier: {s.get('level','?')})"
        for s in bad
    )
    # Build a compact key→name reference so the model can pick the right one
    key_lines = []
    for d in brief.get("l0_domains", []):
        key_lines.append(f"  {d['key']}  \"{d['name']}\"  (L0 domain)")
    for c in brief.get("l1_clusters", []):
        key_lines.append(f"  {c['key']}  \"{c['name']}\"  (L1 cluster)")
    for s in brief.get("l1_singles", []):
        key_lines.append(f"  {s['key']}  \"{s['name']}\"  (L1 single)")
    for c in brief.get("l2_clusters", []):
        key_lines.append(
            f"  {c['key']}  \"{c['name']}\"  (L2 cluster, parent: {c['l1_name']})"
        )
    return CLARIFY_TEMPLATE.format(
        n=len(bad),
        bad_lines=bad_lines,
        valid_key_lines="\n".join(key_lines),
    )


def call_llm(
    journal_id: int, journal_name: str, brief: dict, max_clarify: int = 2
) -> dict:
    """
    Call the LLM for one journal.
    If any returned keys are not in the brief, enter a clarification loop
    (up to max_clarify rounds) before falling back to dropping bad keys.
    """
    user_text = format_brief(brief, journal_name)
    system = SYSTEM_PROMPT.format(max_core=MAX_CORE, max_bleed=MAX_BLEED)
    valid_keys = _valid_keys_from_brief(brief)

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_text},
    ]

    data = {}
    for attempt in range(max_clarify + 1):
        try:
            resp = oai.chat.completions.create(
                model=LLM_MODEL,
                temperature=LLM_TEMP,
                messages=messages,
            )
            raw = resp.choices[0].message.content
            data = parse_llm_response(raw)
        except Exception as exc:
            log.error(
                "LLM parse error for %s (attempt %d): %s",
                journal_name,
                attempt + 1,
                exc,
            )
            break

        # Check for hallucinated keys
        bad = _hallucinated(data.get("core", []), valid_keys) + _hallucinated(
            data.get("bleed", []), valid_keys
        )

        if not bad:
            log.info("  ✓ %s — clean on attempt %d", journal_name, attempt + 1)
            break

        log.warning(
            "  ⚠ %s — %d hallucinated key(s) on attempt %d: %s",
            journal_name,
            len(bad),
            attempt + 1,
            [s.get("key") for s in bad],
        )

        if attempt < max_clarify:
            # Append the assistant turn + a correction prompt and loop
            clarify_msg = _build_clarify_msg(data, valid_keys, brief)
            messages.append({"role": "assistant", "content": raw})
            messages.append({"role": "user", "content": clarify_msg})
        else:
            # Exhausted clarification attempts — log and keep what's valid
            log.error(
                "  ✗ %s — still hallucinating after %d clarification(s); "
                "bad keys will be dropped by _filter_valid",
                journal_name,
                max_clarify,
            )

    data["_journal_id"] = journal_id
    data["_journal_name"] = journal_name
    return data

## 9. Run LLM for all communities

One LLM call per community. The loop logs progress and token usage.  
Typical cost: ~$0.01–0.05 per community depending on the number of candidates.

In [40]:
# # ── Run LLM for all journals ─────────────────────────────────────────────
# # Pull journal names
# journals_df = bq.query(
#     f"""
#     SELECT CAST(JournalId AS INT64) AS journal_id, DisplayName AS journal_name
#     FROM `{TBL_JOURNAL}`
#     WHERE PublisherId = {FRONTIERS_PUBLISHER_ID}
# """
# ).to_dataframe()
# jname_map: dict[int, str] = dict(
#     zip(journals_df["journal_id"], journals_df["journal_name"])
# )

# llm_results: dict[int, dict] = {}
# total_in_tokens = 0
# total_out_tokens = 0

# for i, jid in enumerate(journal_ids):
#     jname = jname_map.get(jid, str(jid))
#     brief = briefs[jid]

#     # Skip if no candidates were found (journal has no taxonomy signal)
#     if not (
#         brief["l0_domains"]
#         or brief["l1_clusters"]
#         or brief["l1_singles"]
#         or brief.get("l2_clusters")
#     ):
#         log.warning(
#             "[%d/%d] %s — no candidates, skipping LLM", i + 1, len(journal_ids), jname
#         )
#         llm_results[jid] = {
#             "core": [],
#             "bleed": [],
#             "match_mode": "none",
#             "confidence": "low",
#             "overall_reasoning": "no candidates",
#             "_journal_id": jid,
#             "_journal_name": jname,
#             "_input_tokens": 0,
#             "_out_tokens": 0,
#         }
#         continue

#     log.info("[%d/%d] %s …", i + 1, len(journal_ids), jname)
#     result = call_llm(jid, jname, brief)
#     llm_results[jid] = result
#     total_in_tokens += result.get("_input_tokens", 0)
#     total_out_tokens += result.get("_out_tokens", 0)

# log.info(
#     "LLM complete. Total tokens in=%d out=%d (≈$%.2f at gpt-4o rates)",
#     total_in_tokens,
#     total_out_tokens,
#     total_in_tokens / 1e6 * 2.5 + total_out_tokens / 1e6 * 10,
# )

## THIS DIDNT WORK FOR MY COMMUNITIES SO IM CHANGING IT
# ── Run LLM for all communities ─────────────────────────────────────────────
llm_results: dict[int, dict] = {}
total_in_tokens = 0
total_out_tokens = 0

for i, jid in enumerate(journal_ids):
    jname = community_names.get(jid, str(jid))  # ← use community_names, not jname_map
    brief = briefs[jid]

    # Skip if no candidates were found (community has no taxonomy signal)
    if not (
        brief["l0_domains"]
        or brief["l1_clusters"]
        or brief["l1_singles"]
        or brief.get("l2_clusters")
    ):
        log.warning(
            "[%d/%d] %s — no candidates, skipping LLM", i + 1, len(journal_ids), jname
        )
        llm_results[jid] = {
            "core": [],
            "bleed": [],
            "match_mode": "none",
            "confidence": "low",
            "overall_reasoning": "no candidates",
            "_journal_id": jid,
            "_journal_name": jname,
            "_input_tokens": 0,
            "_out_tokens": 0,
        }
        continue

    log.info("[%d/%d] %s …", i + 1, len(journal_ids), jname)
    result = call_llm(jid, jname, brief)
    llm_results[jid] = result
    total_in_tokens += result.get("_input_tokens", 0)
    total_out_tokens += result.get("_out_tokens", 0)

log.info(
    "LLM complete. Total tokens in=%d out=%d (≈$%.2f at gpt-4o rates)",
    total_in_tokens,
    total_out_tokens,
    total_in_tokens / 1e6 * 2.5 + total_out_tokens / 1e6 * 10,
)

12:40:19 INFO     [1/1] Cluster 0 …
12:40:38 INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
12:40:38 INFO       ✓ Cluster 0 — clean on attempt 1
12:40:38 INFO     LLM complete. Total tokens in=0 out=0 (≈$0.00 at gpt-4o rates)


## 10. Post-process and flatten to df_out

Identical to nb0006b Stage 7:
1. **Validate keys** — any key the LLM returned that wasn't in the brief is dropped
2. **Deduplicate core** — selections whose L2 territory is already covered by an  
   earlier selection are removed
3. **Deduplicate bleed** — same, but also cannot overlap with core
4. **Compute metrics** per surviving selection:  
   - `n_l2_covered`: number of in-scope L2 topics the cluster covers for this community  
   - `l2_coverage`: `n_l2_covered / total_in_scope_l2`  
   - `article_share`: fraction of community papers inside the cluster

The final `df_out` has one row per (community, tier, cluster_rank).

In [41]:
def _valid_candidate_keys(brief: dict) -> set[str]:
    """Return all valid keys that were presented to the LLM for this journal."""
    keys = set()
    for d in brief.get("l0_domains", []):
        keys.add(d["key"])
    for c in brief.get("l1_clusters", []):
        keys.add(c["key"])
    for s in brief.get("l1_singles", []):
        keys.add(s["key"])
    for c in brief.get("l2_clusters", []):
        keys.add(c["key"])
    return keys


def _filter_valid(
    selections: list[dict], valid_keys: set[str], journal_name: str, tier: str
) -> list[dict]:
    """Drop any LLM-returned selections whose key was not in the brief."""
    clean = []
    for s in selections:
        if s.get("key") in valid_keys:
            clean.append(s)
        else:
            log.warning(
                "[%s] LLM hallucinated %s key '%s' — dropping",
                journal_name,
                tier,
                s.get("key"),
            )
    return clean


def l2_keys_for_candidate(key: str, journal_id: int) -> frozenset[str]:
    """Return the set of in-scope L2 keys for a given candidate key.
    Returns an empty frozenset (rather than raising) for any malformed key.
    """
    try:
        if key.startswith("l0_"):
            l0_txr = int(key[3:])  # raises ValueError if LLM invented a slug
            l1s_in_l0 = l0_members.get(l0_txr, {}).get("l1s", set())
            return frozenset(
                df_scope[
                    (df_scope["journal_id"] == journal_id)
                    & (df_scope["l1_taxref"].isin(l1s_in_l0))
                ]["l2_key"]
            )
        elif key.startswith("l1c_"):
            mems = l1c_members.get(key, frozenset())
            return frozenset(
                df_scope[
                    (df_scope["journal_id"] == journal_id)
                    & (df_scope["l1_taxref"].isin(mems))
                ]["l2_key"]
            )
        elif key.startswith("l2c_"):
            mems = l2c_members.get(key, frozenset())
            return frozenset(
                df_scope[
                    (df_scope["journal_id"] == journal_id)
                    & (df_scope["l2_key"].isin(mems))
                ]["l2_key"]
            )
        elif key.startswith("l1_"):
            l1_txr = int(key[3:])
            return frozenset(
                df_scope[
                    (df_scope["journal_id"] == journal_id)
                    & (df_scope["l1_taxref"] == l1_txr)
                ]["l2_key"]
            )
    except (ValueError, KeyError):
        pass
    return frozenset()


def deduplicate_selections(
    selections: list[dict],
    journal_id: int,
    covered_so_far: frozenset[str] = frozenset(),
) -> list[dict]:
    """
    Remove any selection whose L2 territory is already fully covered by an
    earlier selection.  Works for both core and bleed passes.
    Each surviving selection is annotated with its marginal L2 contribution.
    """
    clean = []
    covered = set(covered_so_far)
    for sel in selections:
        l2s = l2_keys_for_candidate(sel["key"], journal_id)
        marginal = l2s - covered
        if not marginal:
            continue  # entirely subsumed — drop it
        sel = dict(sel)
        sel["_l2_set"] = l2s
        sel["_marginal_l2"] = len(marginal)
        covered |= l2s
        clean.append(sel)
    return clean, frozenset(covered)


LEVEL_NORM = {
    "l0": "l0_domain",
    "l0_domain": "l0_domain",
    "l0 domain": "l0_domain",
    "l1c": "l1_cluster",
    "l1_cluster": "l1_cluster",
    "l1 cluster": "l1_cluster",
    "l1": "l1_single",
    "l1_single": "l1_single",
    "l1 single": "l1_single",
    "l2c": "l2_cluster",
    "l2_cluster": "l2_cluster",
    "l2 cluster": "l2_cluster",
    "l2": "l2_cluster",
}


def normalise_level(raw: str) -> str:
    return LEVEL_NORM.get(raw.lower().replace("-", "_").strip(), raw)


def compute_metrics(sel: dict, journal_id: int, total_scope_l2: int) -> dict:
    """Attach coverage and article-share metrics to a selection."""
    l2s = sel.get("_l2_set", l2_keys_for_candidate(sel["key"], journal_id))
    jscope = df_scope[df_scope["journal_id"] == journal_id]
    in_sel = jscope[jscope["l2_key"].isin(l2s)]
    n_arts_in = int(in_sel["n_articles"].sum())
    total_arts = profiles.get(journal_id, {}).get("n_articles", 1)
    return {
        "n_l2_covered": len(l2s),
        "l2_coverage": round(len(l2s) / max(total_scope_l2, 1), 4),
        "article_share": round(n_arts_in / max(total_arts, 1), 4),
    }


# ── Flatten results into output rows ──────────────────────────────────────
output_rows = []

for jid, res in llm_results.items():
    jname = community_names.get(jid, str(jid))
    total_scope = int(df_scope[df_scope["journal_id"] == jid].shape[0])
    total_arts = profiles.get(jid, {}).get("n_articles", 0)

    # Validate: drop any keys the LLM invented that weren't in the brief
    valid_keys = _valid_candidate_keys(briefs[jid])

    # Deduplicate core
    core_raw = _filter_valid(res.get("core", []), valid_keys, jname, "core")
    core_clean, cov_core = deduplicate_selections(core_raw, jid)

    # Deduplicate bleed (cannot overlap with core OR with each other)
    bleed_raw = _filter_valid(res.get("bleed", []), valid_keys, jname, "bleed")
    bleed_clean, cov_bleed = deduplicate_selections(
        bleed_raw, jid, covered_so_far=cov_core
    )

    match_mode = res.get("match_mode", "combination")
    confidence = res.get("confidence", "low")
    reasoning = res.get("overall_reasoning", "")

    for rank, sel in enumerate(core_clean, start=1):
        m = compute_metrics(sel, jid, total_scope)
        output_rows.append(
            {
                "community_id": jid,
                "community_name": jname,
                "n_community_papers": total_arts,
                "tier": "core",
                "cluster_rank": rank,
                "cluster_key": sel["key"],
                "cluster_name": sel["name"],
                "cluster_level": normalise_level(sel.get("level", "")),
                "n_l2_covered": m["n_l2_covered"],
                "l2_coverage": m["l2_coverage"],
                "article_share": m["article_share"],
                "match_mode": match_mode,
                "llm_confidence": confidence,
                "llm_rationale": sel.get("rationale", ""),
                "llm_reasoning": reasoning,
                "run_date": RUN_DATE,
            }
        )

    for rank, sel in enumerate(bleed_clean, start=1):
        m = compute_metrics(sel, jid, total_scope)
        output_rows.append(
            {
                "community_id": jid,
                "community_name": jname,
                "n_community_papers": total_arts,
                "tier": "bleed",
                "cluster_rank": rank,
                "cluster_key": sel["key"],
                "cluster_name": sel["name"],
                "cluster_level": normalise_level(sel.get("level", "")),
                "n_l2_covered": m["n_l2_covered"],
                "l2_coverage": m["l2_coverage"],
                "article_share": m["article_share"],
                "match_mode": match_mode,
                "llm_confidence": confidence,
                "llm_rationale": sel.get("rationale", ""),
                "llm_reasoning": reasoning,
                "run_date": RUN_DATE,
            }
        )

df_out = pd.DataFrame(output_rows)
log.info(
    "Output: %d rows  (%d journals)", len(df_out), df_out["community_id"].nunique()
)
log.info("Tier breakdown:\n%s", df_out["tier"].value_counts().to_string())

12:40:38 INFO     Output: 5 rows  (1 journals)
12:40:38 INFO     Tier breakdown:
tier
core     3
bleed    2


## 11. Inspect results

In [42]:
# Summary
print(f"Communities processed: {df_out['community_id'].nunique()}")
print(f"Total cluster assignments: {len(df_out)}")
print()
print(
    df_out.groupby(["community_name", "tier"])["cluster_name"]
    .apply(lambda x: ", ".join(x))
    .rename("clusters")
    .to_string()
)

Communities processed: 1
Total cluster assignments: 5

community_name  tier 
Cluster 0       bleed       Comprehensive Psychology, Clinical Specialties
                core     Comprehensive Medicine, Immunology and Autoimm...


In [43]:
# Full table — filter / save as needed
# e.g. df_out.to_csv("community_cluster_assignments.csv", index=False)
df_out[(df_out["tier"] == "core") & (df_out["cluster_rank"] == 1)]

# df_out.to_csv("community_cluster_assignments.csv", index=False)

,community_id,community_name,n_community_papers,tier,cluster_rank,cluster_key,cluster_name,cluster_level,n_l2_covered,l2_coverage,article_share,match_mode,llm_confidence,llm_rationale,llm_reasoning,run_date
0,0,Cluster 0,55994,core,1,l1c_cf6d129d,Comprehensive Medicine,l1_cluster,276,0.1476,0.4873,combination,high,"Covers a wide range of medical disciplines, re...",The community's publication data indicates a s...,2026-06-17


## Trying to assing journals to clusters based on this citation network

In [52]:
import pandas as pd

TBL_CLASSIF = f"{BQ_SRC_DATASET}.classification_raw_20260617_081903"
TBL_PUB_META = f"{BQ_SRC_DATASET}.pub_metadata_raw_20260617_081904"
# Load classification from BigQuery
papers = bq_src.query(
    f"""
        SELECT 
        c.int_id,
        c.micro,
        c.meso,
        c.macro,
        m.pub_id,
        m.is_frontiers,
        m.journal,
        m.date,
        m.title
    FROM `{TBL_CLASSIF}` c
    JOIN `{TBL_PUB_META}` m
    ON c.int_id = m.int_id
"""
).to_dataframe()


# Get primary taxonomy assignment per cluster (core rank 1)
primary = df_out[(df_out["tier"] == "core") & (df_out["cluster_rank"] == 1)][
    ["community_id", "cluster_name", "cluster_level", "article_share"]
].rename(columns={"cluster_name": "taxonomy_name"})

# Join papers to taxonomy via macro cluster
papers_with_taxonomy = papers.merge(
    primary, left_on="meso", right_on="community_id", how="left"
)

# Result: each paper now has its taxonomy assignment
papers_with_taxonomy[["pub_id", "journal", "title", "macro", "taxonomy_name"]].head(10)

c:\Users\sophie.wilson\AppData\Local\miniconda3\envs\scope_drift\lib\site-packages\google\cloud\bigquery\table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,pub_id,journal,title,macro,taxonomy_name
0,813,Vaccine,"CD40L expressed from the canarypox vector, ALV...",0,Comprehensive Medicine
1,2625,Alimentary Pharmacology & Therapeutics,Meta‐analysis: Smectite in the treatment of ac...,0,Comprehensive Medicine
2,5994,Glycobiology,Two types of galactosylated fucose motifs are ...,0,Comprehensive Medicine
3,6537,The Oncologist,Corticosteroid Use in Patients with Glioblasto...,0,Comprehensive Medicine
4,12024,Proceedings of the National Academy of Sciences,Interindividual variation in human T regulator...,0,Comprehensive Medicine
5,16743,European Journal of Clinical Nutrition,Impact of consuming a milk drink containing a ...,0,Comprehensive Medicine
6,16993,Cancer,"Prognostic significance of MYC, BCL2, and BCL6...",0,Comprehensive Medicine
7,17903,The Journal of Experimental Medicine,The lineage-defining factors T-bet and Bcl-6 c...,0,Comprehensive Medicine
8,18016,Annual Review of Medicine,Pathogenesis of Macrophage Activation Syndrome...,0,Comprehensive Medicine
9,18085,Behaviour Research and Therapy,A short form of the metacognitions questionnai...,0,Comprehensive Medicine


In [53]:
jnl_counts = (
    papers_with_taxonomy[
        papers_with_taxonomy["journal"].isin(
            [
                "Frontiers in Immunology",
                "Frontiers in Public Health",
                "Frontiers in Medicine",
                "Frontiers in Oncology",
                "Frontiers in Psychology",
            ]
        )
    ]
    .groupby(["journal", "taxonomy_name"])
    .size()
    .reset_index(name="n_papers")
)

# Get the top taxonomy for each journal
top_taxonomy_per_journal = (
    jnl_counts.sort_values("n_papers", ascending=False)
    .drop_duplicates("journal")
    .sort_values("n_papers", ascending=False)
)
top_taxonomy_per_journal

,journal,taxonomy_name,n_papers
0,Frontiers in Immunology,Comprehensive Medicine,3285
3,Frontiers in Psychology,Comprehensive Medicine,1715
1,Frontiers in Medicine,Comprehensive Medicine,1537
2,Frontiers in Oncology,Comprehensive Medicine,1520
4,Frontiers in Public Health,Comprehensive Medicine,1367
